# Experiments

This file contains multiple experiments that were done in order to find the solution to the [task](task.md). Most of the necesary code is located in [solution.py](solution.py).

## Preparation

First we load and prepare the data. Preparation includes:
- removing whitespaces
- assigning labels for cheap, average and expensive houses
- converting categorical data to numerical values

In [2]:
from solution import prepare_train_data, NetConfig, NeuralNetwork
import numpy as np

X, y = prepare_train_data("train_data.csv")
X, y = np.array(X), np.array(y)

Below are the methods we will use to compare the quality of predictions of created models.
- `calc_accuracy` measures average accuracy across all price categories (in our case: 0, 1, 2)
- `evaluate_cv` estimates model performance using 5-fold stratified cross-validation

In [3]:
from sklearn.model_selection import StratifiedKFold
import copy

def calc_accuracy(preds, targets) -> float:
    return np.mean([
        (preds[targets == i] == i).mean()
        for i in range(3)
        if (targets == i).sum() > 0
    ])


def evaluate_cv(name: str, model, X, y) -> None:
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, te in kf.split(X, y):
        m = copy.deepcopy(model)
        m.fit(X[tr], y[tr])
        scores.append(calc_accuracy(m.predict(X[te]), y[te]))
    
    print(f"{name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## Experiments

### 1. Managing unbalanced dataset
Since the task specifies that the dataset favours certain house classes and is unbalanced, we need to find a way to make our model prepared for that.

In [4]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

model1 = NeuralNetwork(NetConfig(sampler=None))
evaluate_cv("No balancing", model1, X, y)

model2 = NeuralNetwork(NetConfig(sampler=SMOTE))
evaluate_cv("SMOTE", model2, X, y)

model3 = NeuralNetwork(NetConfig(sampler=RandomUnderSampler))
evaluate_cv("RandomUnderSampler", model3, X, y)

model4 = NeuralNetwork(NetConfig(sampler=RandomOverSampler))
evaluate_cv("RandomOverSampler", model4, X, y)

model5 = NeuralNetwork(NetConfig(sampler=None, class_weight=True))
evaluate_cv("Class weights", model5, X, y)


No balancing: 0.7492 ± 0.0203
SMOTE: 0.8752 ± 0.0036
RandomUnderSampler: 0.8773 ± 0.0047
RandomOverSampler: 0.8786 ± 0.0069
Class weights: 0.8826 ± 0.0043


We can see that **balancing the dataset does improve the results**. 
- Without any balancing the model scores  **0.749**, while every balancing method jumps to around **0.87–0.88**. 
- The best result comes from **Class Weight (0.8826 ± 0.0043)**, which we will use as the default for all further experiments.

### 2. Network layers

### 3. Adding Batch Normalization

### 4. Different Dropouts

## Conclusions